# Avance 3. Integracion, analisis descriptivo y ETL reproducible

**Proyecto:** SuperMarketMundial · **Equipo:** Equipo BI SuperMarket
**Integrantes:** Francisco Aramis Alcantara Vivas · Brayan Perez Nava · Joseline Ariana Marcial Espinoza

En este notebook juntamos lo que hicimos en el avance 2 (leer, limpiar y revisar los datos), pero ahora como un proceso que se puede repetir completo. Cada paso es una funcion, y cada vez que se corre el notebook se vuelven a crear los archivos limpios y la bitacora desde el Excel original. Para correrlo solo hay que darle en **Entorno de ejecucion → Ejecutar todo** en Colab, o en **Run All** en VS Code.

El perfilamiento ya lo hicimos en `notebooks/avance2_preparacion_datos.ipynb`, por eso aqui no lo repetimos

## 3.1 Continuidad con los avances anteriores

**Pregunta de negocio.** ¿A partir de que nivel de descuento la venta deja de ser rentable, y ese punto es el mismo en todas las categorias, subcategorias y mercados?

**Decision.** Queremos ayudar a la Gerencia de Marketing a poner un descuento maximo por subcategoria y mercado, para que dejen de vender con perdida.

**Fuente.** El archivo *Ventas Supermercado* que publico Rodrigo Ayala en GitHub (https://github.com/RodrigoAyalaDA/Analisis-Ventas-Excel). Es un Excel con seis hojas. Usamos la hoja *Ventas Supermercado*, y las hojas *Regiones* y *Productos* solo para revisar.

**Columnas que necesitamos.** Descuento, Ventas, Utilidad, Categoria, Subcategoria, Mercado, Fecha Salida y Costo Envío.

**Con que datos trabajamos ahora.** Ventas con descuento de US, LATAM, EU y APAC, de enero de 2011 a diciembre de 2014.

**Cambios que hicimos por la retroalimentacion**

| ¿Que nos dijeron? | ¿Quien y cuando? | ¿Que cambiamos? | ¿Donde se ve? |
|---|---|---|---|
| Nos faltaban subpreguntas | Profesora, avance 1 | Agregamos cinco subpreguntas | Avance 1 corregido, seccion 3.3 |
| Comparar zonas con un numero parecido de ventas y ver los cambios por mes | Profesora, avance 1 | Nos quedamos con cuatro mercados (regla R4) y creamos la columna Mes | Secciones 3.3 y 3.5 |
| Ver tambien la subcategoria | Profesora, avance 1 | La subcategoria esta en los datos limpios y en la estadistica | Seccion 3.5 |
| Los 629 descuentos con tres decimales | Profesora, 3 de noviembre | Los redondeamos a dos decimales (regla R1) | Seccion 3.3 |
| Las parejas con el mismo pedido y producto | Profesora, 3 de noviembre | Las juntamos en una sola fila sumando cantidad, ventas, utilidad y envio (regla R3) | Seccion 3.3 |
| Las 3 parejas con cliente distinto | Profesora, revision del avance 2 | Ya no las dejamos pendientes, ahora las quitamos (regla R2) | Seccion 3.3 |
| Trabajar solo con ventas que tienen descuento | Profesora, revision del avance 2 | Quitamos las ventas sin descuento (regla R5) | Seccion 3.3 |
| Dejar solo las reglas de negocio | Profesora, revision del avance 2 | Lo tecnico (espacios, mayusculas y tipos) lo hace la funcion `normalizar()` y queda en la bitacora, pero ya no lo contamos como regla | Seccion 3.3 |
| El costo de envio lo paga el cliente | Profesora, revision del avance 2 | No lo restamos de la utilidad, asi que la Utilidad es la utilidad neta de la tienda (regla R6) | Seccion 3.3, paso 4 |

### Preparar el entorno

Primero ubicamos las carpetas del proyecto y el Excel. Si se abre en Colab y el Excel no esta en `data/raw/`, el notebook lo pide para subirlo y lo guarda ahi tal como llega.

In [ ]:
import hashlib                 # para sacar la huella del archivo
from datetime import datetime  # para poner la fecha en la bitacora
from pathlib import Path       # para manejar las rutas de las carpetas

import pandas as pd

pd.set_option("display.max_columns", None)  # para que se vean todas las columnas al imprimir

# Buscamos la carpeta principal del proyecto. Si abrimos el notebook desde la carpeta notebooks/,
# la carpeta principal es la de arriba. Si estamos en Colab usamos la carpeta donde estamos.
if (Path.cwd() / "data").exists():
    RAIZ = Path.cwd()
elif (Path.cwd().parent / "data").exists():
    RAIZ = Path.cwd().parent
else:
    RAIZ = Path.cwd()

CARPETA_CRUDOS = RAIZ / "data" / "raw"          # aqui esta el Excel original
CARPETA_SALIDAS = RAIZ / "data" / "processed"   # aqui se guardan los archivos limpios
CARPETA_BITACORA = RAIZ / "bitacora"            # aqui se guarda bitacora.csv
for carpeta in (CARPETA_CRUDOS, CARPETA_SALIDAS, CARPETA_BITACORA):
    carpeta.mkdir(parents=True, exist_ok=True)  # si no existen las creamos

# Buscamos el Excel en data/raw. Si no esta (por ejemplo en Colab), lo pedimos y lo guardamos ahi sin cambiarlo
excel_en_raw = sorted(CARPETA_CRUDOS.glob("*.xlsx"))
if not excel_en_raw:
    from google.colab import files
    subido = files.upload()
    nombre = next(iter(subido))
    (CARPETA_CRUDOS / nombre).write_bytes(subido[nombre])
    excel_en_raw = sorted(CARPETA_CRUDOS.glob("*.xlsx"))
ENTRADA = excel_en_raw[0]

print("Version de pandas:", pd.__version__)
print("Archivo de entrada:", ENTRADA.relative_to(RAIZ))

Version de pandas: 2.2.3
Archivo de entrada: data/raw/Ventas Supermercado- Rodrigo Ayala.xlsx


### Valores que usamos en todo el proceso

Juntamos aqui los valores que usan las reglas. Asi, si algo cambia (por ejemplo, si quisieramos agregar otro mercado), solo se cambia en un lugar.

In [ ]:
HUELLA_REGISTRADA = "a599dc2cf5d00385"   # inicio de la huella que registramos en el avance 2
HOJA_VENTAS = "Ventas Supermercado"

# Las 35 columnas que debe traer la hoja de ventas
COLUMNAS_ENTRADA = [
    "Número Venta", "ID Venta", "Mes Salida", "Dia Salida", "Año Salida", "Fecha Salida",
    "Dia Semana Fecha Salida", "Mes Entrega", "Dia Entrega", "Año Entrega", "Fecha Entrega",
    "Dia Semana Fecha Entrega", "Dias Transcurridos", "Dias laborables transcurridos", "Método Envio",
    "ID Cliente", "Número Cliente", "Nombre Cliente", "Segmento", "Ciudad", "Estado", "País",
    "Mercado", "Region", "ID Producto", "Categoria", "Subcategoria", "Nombre del producto",
    "Ventas", "Cantidad", "Descuento", "Utilidad", "Ganancia", "Costo Envío", "Prioridad Envio",
]
CLAVE = ["ID Venta", "ID Producto"]                  # una fila es un producto dentro de un pedido
CODIGOS = ["ID Venta", "ID Cliente", "ID Producto"]
FECHAS = ["Fecha Salida", "Fecha Entrega"]
NUMEROS = ["Cantidad", "Ventas", "Descuento", "Utilidad", "Costo Envío"]
MERCADOS_ALCANCE = ["US", "LATAM", "EU", "APAC"]     # regla R4

# Tramos de descuento (los del avance 1, sin el de 0 % porque la regla R5 deja solo ventas con descuento)
LIMITES_TRAMO = [0, 0.10, 0.20, 0.30, 0.40, 0.50, 1.00]
NOMBRES_TRAMO = ["1 a 10 %", "11 a 20 %", "21 a 30 %", "31 a 40 %", "41 a 50 %", "Mas de 50 %"]

# Las 23 columnas que salen en el archivo limpio
COLUMNAS_SALIDA = [
    "ID Venta", "ID Producto", "ID Cliente", "Fecha Salida", "Fecha Entrega", "Mes",
    "Mercado", "País", "Segmento", "Categoria", "Subcategoria", "Nombre del producto",
    "Método Envio", "Cantidad", "Ventas", "Descuento", "Descuento original", "Tramo de descuento",
    "Utilidad", "Margen", "Ganancia", "Costo Envío", "Número de envíos",
]
RESPONSABLE = "Equipo BI SuperMarket"   # quien corre el proceso, se puede cambiar por un nombre

### Funcion anotar()

Esta funcion nos sirve para hacer la  bitacora automaticamente. La sacamos de la Nota 11 y le agregamos los campos que pide la guia (responsable, regla, resultado y observaciones). Cada paso del proceso la llama cuando termina, asi la bitacora se llena sola y no la escribimos a mano. La tabla completa la mostramos en la seccion 3.6.

In [ ]:
bitacora = []   # aqui se van guardando los renglones de la bitacora

def anotar(paso, antes, despues, detalle, regla, resultado, observaciones="", responsable=RESPONSABLE):
    """Agrega un renglon a la bitacora con lo que hizo un paso."""
    bitacora.append({
        "fecha": datetime.now().astimezone().isoformat(timespec="seconds"),  # cuando se hizo
        "archivo": ENTRADA.name,        # con que archivo
        "paso": paso,                   # que paso fue
        "filas_antes": antes,           # cuantas filas habia antes
        "filas_despues": despues,       # cuantas quedaron despues
        "detalle": detalle,             # que cambio
        "responsable": responsable,
        "regla": regla,                 # que regla se aplico
        "resultado": resultado,         # "Correcto" o "Con observaciones"
        "observaciones": observaciones,
    })

## 3.2 Fuentes e integracion

Antes de limpiar queremos dejar claro de donde vienen los datos y si necesitamos juntarlos con otras tablas.

| Pregunta | Respuesta |
|---|---|
| Cuantas fuentes hay? | Una sola, el Excel de `data/raw/`. No nos llegan archivos nuevos |
| Como sabemos que es el mismo archivo? | Con la huella SHA-256, que empieza con `a599dc2cf5d00385`, igual que en el avance 2. La funcion `extraer()` la vuelve a sacar cada vez |
| Que representa una fila? | Un producto dentro de un pedido, como cada linea de un ticket. La identificamos con `ID Venta` + `ID Producto` |
| Juntamos tablas? | No. La hoja de ventas ya trae la categoria, la subcategoria, el pais y el mercado. Las hojas *Regiones* y *Productos* solo las usamos para revisar que cada pais y producto exista |
| Que columnas nuevas hacemos? | Mes, Tramo de descuento y Margen, para poder agrupar por tramo, categoria, subcategoria y mercado |

### Paso 1. Extraer

`extraer()` saca la huella del archivo, lee las tres hojas y revisa que la hoja de ventas traiga las 35 columnas que esperamos. Si falta o sobra una columna, el proceso se detiene y dice cual. Si la huella no es la misma, sigue, pero lo anota en la bitacora como "Con observaciones", porque puede ser otra version del archivo y hay que revisarlo.

In [ ]:
def extraer(ruta):
    """Lee la hoja de ventas y los catalogos, y revisa la huella y las columnas."""
    huella = hashlib.sha256(Path(ruta).read_bytes()).hexdigest()  # huella del archivo
    libro = pd.ExcelFile(ruta)
    ventas = libro.parse(HOJA_VENTAS)      # la hoja principal
    regiones = libro.parse("Regiones")     # catalogos, solo para revisar
    productos = libro.parse("Productos")

    # Revisamos que no falte ni sobre ninguna columna
    faltan = set(COLUMNAS_ENTRADA) - set(ventas.columns)
    sobran = set(ventas.columns) - set(COLUMNAS_ENTRADA)
    if faltan or sobran:
        raise ValueError(f"Revisa las columnas del archivo. Faltan: {faltan}. Sobran: {sobran}")

    # Comparamos la huella con la que registramos en el avance 2
    misma_version = huella.startswith(HUELLA_REGISTRADA)
    anotar("extraer", len(ventas), len(ventas),
           f"Hoja {HOJA_VENTAS} con {ventas.shape[1]} columnas; ventas del "
           f"{ventas['Fecha Salida'].min():%Y-%m-%d} al {ventas['Fecha Salida'].max():%Y-%m-%d}",
           regla="Revisar la huella y que esten las 35 columnas",
           resultado="Correcto" if misma_version else "Con observaciones",
           observaciones=("Es la misma version del avance 2" if misma_version
                          else "La huella no coincide con la del avance 2, hay que revisar si es otra version"))
    return ventas[COLUMNAS_ENTRADA], regiones, productos, huella


crudos, regiones, productos, huella_entrada = extraer(ENTRADA)

# Imprimimos lo que leimos para comprobarlo
print("Huella SHA-256:", huella_entrada)
print("Es la misma version del avance 2?", huella_entrada.startswith(HUELLA_REGISTRADA))
print("Ventas:", crudos.shape, "| Regiones:", regiones.shape, "| Productos:", productos.shape)
print("Ventas del", crudos["Fecha Salida"].min().date(), "al", crudos["Fecha Salida"].max().date())
print("Mercados:", crudos["Mercado"].nunique(), "| Paises:", crudos["País"].nunique())

Huella SHA-256: a599dc2cf5d00385b9fae82204c373353d22140a26762e2acf36a35beb6c42e3
Es la misma version del avance 2? True
Ventas: (51290, 35) | Regiones: (152, 3) | Productos: (10768, 4)
Ventas del 2011-01-01 al 2014-12-31
Mercados: 7 | Paises: 147


### ¿Que representa una fila antes de limpiar?

Como vimos en el avance 2, un pedido puede tener varios productos, asi que `ID Venta` solo no sirve para identificar una fila. Usamos `ID Venta` + `ID Producto`, que casi siempre es unica. Las 38 parejas que se repiten las arreglamos con las reglas R2 y R3.

In [ ]:
print("Filas:", len(crudos))
print("Pedidos distintos (ID Venta):", crudos["ID Venta"].nunique())                       # cuantos tickets hay
print("Combinaciones distintas de pedido y producto:", len(crudos[CLAVE].drop_duplicates()))
print("Filas que repiten pedido y producto:", crudos.duplicated(subset=CLAVE, keep=False).sum())

Filas: 51290
Pedidos distintos (ID Venta): 25086
Combinaciones distintas de pedido y producto: 51252
Filas que repiten pedido y producto: 76


### Por que no juntamos las hojas de catalogo?

Segun la Nota 09 (seccion 5), para agregarle columnas de un catalogo a las ventas sin que se multipliquen las filas, el catalogo debe tener una sola fila por clave. Aqui revisamos si *Regiones* y *Productos* cumplen eso. Usamos `validate="many_to_one"`, que marca error si el catalogo repite claves.

In [ ]:
# Contamos cuantas claves se repiten en cada catalogo
print("Paises repetidos en Regiones:", regiones["País"].duplicated().sum())
print("Codigos repetidos en Productos:", productos["ID Producto"].duplicated().sum())

# Intentamos el cruce con validate. Si da error es porque el catalogo repite claves,
# y en ese caso contamos cuantas filas saldrian si lo cruzaramos de todos modos
for nombre, catalogo, llave in [("Regiones", regiones, "País"), ("Productos", productos, "ID Producto")]:
    try:
        crudos.merge(catalogo, on=llave, how="left", validate="many_to_one")
        print(f"{nombre}: si se puede cruzar sin multiplicar filas")
    except pd.errors.MergeError:
        filas = len(crudos.merge(catalogo, on=llave, how="left"))
        print(f"{nombre}: repite claves. Si lo cruzaramos, las ventas pasarian de {len(crudos):,} a {filas:,} filas")

Paises repetidos en Regiones: 5
Codigos repetidos en Productos: 476
Regiones: repite claves. Si lo cruzaramos, las ventas pasarian de 51,290 a 81,640 filas
Productos: repite claves. Si lo cruzaramos, las ventas pasarian de 51,290 a 55,996 filas


**Resultado de la integracion:**
* **Juntamos tablas?** No, los dos catalogos repiten claves (Regiones repite 5 paises y Productos 476 codigos), asi que si los cruzaramos las ventas pasarian de 51,290 a 81,640 filas con Regiones y a 55,996 con Productos. Eso inflaria las ventas.
* **Para que los usamos entonces?** Solo para revisar en el paso 5 que cada pais y producto de las ventas exista en ellos.

## 3.3 Contrato y ejecucion del ETL

La guia nos pide hacer un contrato antes de programar, esto es escribir que entra, que le hacemos y que debe salir

| Pregunta | Lo que acordamos |
|---|---|
| Que entra? | El Excel de `data/raw/` con la hoja *Ventas Supermercado* y sus 35 columnas |
| Que le hacemos? |Le hacemos los arreglos tecnicos (espacios, mayusculas y tipos) y le aplicamos las seis reglas de negocio de abajo|
| Que sale? | `data/processed/ventas_preparadas.csv`, con 23 columnas y una fila por pedido + producto sin repetirse. Tambien salen `registros_excluidos.csv`, `registros_pendientes.csv` y `bitacora/bitacora.csv` |
| Que pasa si algo viene mal? | Si falta o sobra una columna, el proceso se detiene. Si un valor no se puede pasar a fecha o a numero, queda vacio, la fila no pasa la revision y se va a `registros_pendientes.csv` con la regla que fallo |

**Reglas de negocio**

| Regla | ¿Que hacemos? | ¿Por que? | ¿Como lo comprobamos? |
|---|---|---|---|
| R1 | Redondeamos a dos decimales los 629 descuentos que tenian tres decimales (0.002, 0.202, 0.402 y 0.602), todos de Copiers en LATAM. El valor original lo guardamos en *Descuento original* | La profesora nos indico tomarlos como error de captura (3 de noviembre) | Deben cambiar exactamente 629 valores y todos deben ser de Copiers en LATAM |
| R2 | Quitamos las 3 parejas con el mismo pedido y producto pero con cliente, ciudad y fecha distintos (6 filas) | No pueden ser el mismo pedido, y la profesora nos indico quitarlas en la revision del avance 2 | Deben salir 6 filas en `registros_excluidos.csv` con su motivo |
| R3 | Juntamos en una fila las 35 parejas que solo cambian en cantidad, ventas, utilidad y costo de envio. Sumamos esas cuatro columnas y agregamos *Número de envíos* | La profesora nos indico tomarlas como un pedido que se mando en dos partes (3 de noviembre) | La clave ya no se repite y los totales no cambian |
| R4 | Nos quedamos solo con US, LATAM, EU y APAC | Son los mercados con un numero parecido de ventas, como nos pidieron en el avance 1 | Las demas filas se van a excluidos con su motivo |
| R5 | Nos quedamos solo con las ventas con descuento mayor a 0, ya redondeado | La profesora nos lo indico en la revision del avance 2 | Las ventas sin descuento se van a excluidos con su motivo |
| R6 | El costo de envio lo paga el cliente, asi que no lo contamos en la utilidad. La Utilidad del Excel ya es la utilidad neta, y el margen sale solo de Utilidad y Ventas | La profesora nos indico tomar el envio como un pago del cliente (revision del avance 2) | Comprobamos que la Utilidad no trae restado el envio (paso 4), y la tabla final revisa que Margen sea igual a Utilidad ÷ Ventas |


**Orden de los pasos:** extraer → normalizar → R1 → R2 → R3 → R4 → R5 → derivar (aqui se aplica R6) → revisar → preparar → guardar. Cada paso es una funcion y anota en la bitacora lo que hizo.

### Paso 2. Normalizar

Aqui hacemos los arreglos tecnicos, que no cambian lo que significan los datos. Quitamos espacios en las orillas de los codigos y de los nombres de producto, pasamos los codigos a mayusculas y nos aseguramos de que las fechas sean fechas y los montos sean numeros. Luego revisamos dos cosas, que al pasar a mayusculas no se junten dos clientes distintos y que ningun valor se haya quedado vacio al convertir.

In [ ]:
def normalizar(tabla):
    """Limpia textos y asegura tipos, sin cambiar lo que significan los datos."""
    resultado = tabla.copy()   # trabajamos sobre una copia, la tabla original no se toca

    # Codigos: quitamos espacios de las orillas y pasamos a mayusculas
    for col in CODIGOS:
        resultado[col] = resultado[col].astype("string").str.strip().str.upper()
    # Nombres de producto: solo quitamos espacios, sin cambiar mayusculas
    resultado["Nombre del producto"] = resultado["Nombre del producto"].astype("string").str.strip()

    # Fechas y montos: los convertimos. Si algo no se puede convertir queda vacio (errors="coerce")
    for col in FECHAS:
        resultado[col] = pd.to_datetime(resultado[col], errors="coerce")
    for col in NUMEROS:
        resultado[col] = pd.to_numeric(resultado[col], errors="coerce")

    # Contamos que cambio para anotarlo en la bitacora
    codigos_cambiados = int(sum((resultado[c] != tabla[c]).sum() for c in CODIGOS))
    nombres_cambiados = int((resultado["Nombre del producto"] != tabla["Nombre del producto"]).sum())
    vacios = int(resultado[FECHAS + NUMEROS].isna().sum().sum())
    # Si el numero de clientes no cambia, no se junto ningun cliente con otro
    clientes_antes, clientes_despues = tabla["ID Cliente"].nunique(), resultado["ID Cliente"].nunique()

    anotar("normalizar", len(tabla), len(resultado),
           f"{codigos_cambiados} codigos y {nombres_cambiados} nombres de producto corregidos; "
           f"{vacios} valores vacios al convertir tipos; clientes distintos {clientes_antes} → {clientes_despues}",
           regla="Quitar espacios, pasar codigos a mayusculas y asegurar tipos",
           resultado="Correcto" if vacios == 0 and clientes_antes == clientes_despues else "Con observaciones")
    return resultado


normalizados = normalizar(crudos)

# Imprimimos cuantos valores cambiaron en cada columna
for col in CODIGOS + ["Nombre del producto"]:
    print(f"{col}: valores que cambiaron:", (normalizados[col] != crudos[col]).sum())
print("Clientes distintos antes y despues:", crudos["ID Cliente"].nunique(), normalizados["ID Cliente"].nunique())
print("Valores que quedaron vacios al convertir:", normalizados[FECHAS + NUMEROS].isna().sum().sum())

ID Venta: valores que cambiaron: 0
ID Cliente: valores que cambiaron: 139
ID Producto: valores que cambiaron: 0
Nombre del producto: valores que cambiaron: 16
Clientes distintos antes y despues: 1590 1590
Valores que quedaron vacios al convertir: 0


### Paso 3. Reglas de negocio

Cada regla es una funcion, y al terminar anota en la bitacora cuantas filas habia antes y despues.

#### R1. Redondear los descuentos con tres decimales

In [ ]:
def r1_redondear_descuentos(tabla):
    """R1. Guarda el descuento como venia y lo redondea a dos decimales."""
    resultado = tabla.copy()
    resultado["Descuento original"] = resultado["Descuento"]          # guardamos como venia
    resultado["Descuento"] = resultado["Descuento"].round(2)          # redondeamos

    # Contamos cuantos cambiaron y de que mercado y subcategoria son
    cambiados = resultado["Descuento"] != resultado["Descuento original"]
    grupos = resultado.loc[cambiados, ["Mercado", "Subcategoria"]].drop_duplicates()
    anotar("R1 redondear descuentos", len(tabla), len(resultado),
           f"{cambiados.sum()} descuentos redondeados a dos decimales",
           regla="R1. Los descuentos con tres decimales se toman como error de captura (indicacion de la profesora)",
           resultado="Correcto",
           observaciones="Todos son Copiers de LATAM" if len(grupos) == 1 else f"{len(grupos)} grupos afectados")
    return resultado


con_r1 = r1_redondear_descuentos(normalizados)

# Revisamos que sean 629 y que todos sean Copiers de LATAM
cambiados = con_r1["Descuento"] != con_r1["Descuento original"]
print("Descuentos redondeados:", cambiados.sum())
print(con_r1.loc[cambiados, ["Descuento original", "Descuento"]].round(3).value_counts())
print(con_r1.loc[cambiados, ["Mercado", "Subcategoria"]].value_counts())

Descuentos redondeados: 629
Descuento original  Descuento
0.002               0.0          461
0.402               0.4          104
0.202               0.2           41
0.602               0.6           23
Name: count, dtype: int64
Mercado  Subcategoria
LATAM    Copiers         629
Name: count, dtype: int64


#### R2. Quitar las parejas con cliente distinto

Buscamos las parejas de pedido + producto donde aparece mas de un cliente. No las borramos sin dejar rastro, las mandamos a la tabla de excluidos con su motivo para poder contarlas en la conciliacion.

In [ ]:
def r2_eliminar_parejas_inconsistentes(tabla):
    """R2. Quita las parejas con el mismo pedido y producto pero con cliente distinto."""
    # Para cada pedido + producto contamos cuantos clientes distintos hay
    clientes_por_clave = tabla.groupby(CLAVE)["ID Cliente"].transform("nunique")
    # Si hay mas de uno, esa pareja no puede ser el mismo pedido
    es_inconsistente = clientes_por_clave > 1

    # Las guardamos aparte con la regla y el motivo
    eliminadas = tabla[es_inconsistente].copy()
    eliminadas["Número de envíos"] = 1   # cada una es una fila del Excel
    eliminadas["Regla"] = "R2"
    eliminadas["Motivo"] = "Mismo pedido y producto con cliente, ciudad y fecha distintos"

    resultado = tabla[~es_inconsistente].copy()
    anotar("R2 quitar parejas con cliente distinto", len(tabla), len(resultado),
           f"{len(eliminadas)} filas de {eliminadas[CLAVE].drop_duplicates().shape[0]} parejas pasan a excluidos",
           regla="R2. Las parejas con cliente distinto no son el mismo pedido (indicacion de la profesora)",
           resultado="Correcto")
    return resultado, eliminadas


con_r2, excluidos_r2 = r2_eliminar_parejas_inconsistentes(con_r1)
print("Filas antes:", len(con_r1), "| despues:", len(con_r2), "| quitadas:", len(excluidos_r2))
display(excluidos_r2[CLAVE + ["ID Cliente", "Ciudad", "Fecha Salida", "Cantidad", "Ventas", "Costo Envío"]])

Filas antes: 51290 | despues: 51284 | quitadas: 6


,ID Venta,ID Producto,ID Cliente,Ciudad,Fecha Salida,Cantidad,Ventas,Costo Envío
19730,FR-2014-1314291,OFF-ST-10004367,ZC-21910,Le Pontet,2014-08-12,5,277.56,13.77
26310,FR-2014-5856066,OFF-BI-10001119,BP-11050,Amiens,2014-09-16,3,17.73,1.11
26705,FR-2014-1314291,OFF-ST-10004367,RB-19330,Nancy,2014-12-16,5,277.56,22.16
34535,FR-2014-5856066,OFF-BI-10001119,SW-20455,Marseille,2014-07-21,1,5.91,0.81
45546,GE-2013-3945862,OFF-AR-10003066,TC-21475,Cologne,2012-12-27,2,52.86,6.08
47325,GE-2013-3945862,OFF-AR-10003066,AA-10480,Leipzig,2012-12-28,5,132.15,9.90


#### R3. Juntar las parejas que son un pedido mandado en dos partes

Antes de juntar revisamos que en cada pareja solo cambien Cantidad, Ventas, Utilidad y Costo Envío. Si cambiara otra columna, quedarnos con el valor de la primera fila seria escoger uno al azar, y en ese caso el proceso se detiene. Despues sumamos esas cuatro columnas, anotamos en *Número de envíos* cuantas filas se juntaron y volvemos a calcular *Ganancia* con la utilidad ya sumada.

In [ ]:
SE_SUMAN = ["Cantidad", "Ventas", "Utilidad", "Costo Envío"]   # columnas que se suman al juntar

def r3_unir_envios(tabla):
    """R3. Junta las parejas de pedido + producto sumando cantidad, ventas, utilidad y envio."""
    resultado = tabla.copy()
    # Contamos cuantas filas tiene cada pedido + producto (1 o 2)
    resultado["Número de envíos"] = resultado.groupby(CLAVE)["ID Venta"].transform("size")

    # Revisamos que en las parejas las demas columnas sean iguales en las dos filas
    parejas = resultado[resultado["Número de envíos"] > 1]
    otras = [c for c in resultado.columns if c not in CLAVE + SE_SUMAN]
    columnas_que_cambian = (parejas.groupby(CLAVE)[otras].nunique(dropna=False) > 1).sum()
    columnas_que_cambian = columnas_que_cambian[columnas_que_cambian > 0]
    if len(columnas_que_cambian) > 0:
        raise ValueError(f"Hay parejas que cambian en otras columnas: {list(columnas_que_cambian.index)}")

    # Juntamos: las cuatro columnas se suman y en las demas nos quedamos con el primer valor (son iguales)
    reglas = {c: ("sum" if c in SE_SUMAN else "first") for c in resultado.columns if c not in CLAVE}
    resultado = resultado.groupby(CLAVE, as_index=False, sort=False).agg(reglas)
    # Ganancia otra vez, ahora con la utilidad sumada
    resultado["Ganancia"] = resultado["Utilidad"].ge(0).map({True: "Ganancia", False: "Perdida"})

    unidas = int((resultado["Número de envíos"] == 2).sum())
    anotar("R3 juntar envios", len(tabla), len(resultado),
           f"{unidas} parejas juntadas; se sumaron {', '.join(SE_SUMAN)} y se recalculo Ganancia",
           regla="R3. Las parejas se toman como un pedido mandado en dos partes (indicacion de la profesora)",
           resultado="Correcto",
           observaciones="En las parejas solo cambian las cuatro columnas que se suman")
    return resultado


con_r3 = r3_unir_envios(con_r2)
print("Filas antes:", len(con_r2), "| despues:", len(con_r3))
print("Parejas juntadas:", (con_r3["Número de envíos"] == 2).sum())
print("Pedido + producto repetidos despues de juntar:", con_r3.duplicated(subset=CLAVE).sum())

Filas antes: 51284 | despues: 51249
Parejas juntadas: 35
Pedido + producto repetidos despues de juntar: 0


#### R4 y R5. Mercados que analizamos y ventas con descuento

Las filas que no entran no se pierden. Se van a la tabla de excluidos con la regla y el motivo.

In [ ]:
def separar(tabla, condicion, regla, motivo, paso, detalle_regla):
    """Deja las filas que cumplen la condicion y manda las demas a excluidos."""
    dentro = tabla[condicion].copy()
    fuera = tabla[~condicion].copy()
    fuera["Regla"] = regla
    fuera["Motivo"] = motivo
    anotar(paso, len(tabla), len(dentro),
           f"{len(fuera)} filas pasan a excluidos ({int(fuera['Número de envíos'].sum())} filas del Excel)",
           regla=detalle_regla, resultado="Correcto")
    return dentro, fuera


# R4: solo los cuatro mercados
con_r4, excluidos_r4 = separar(
    con_r3, con_r3["Mercado"].isin(MERCADOS_ALCANCE), "R4", "Mercado fuera del alcance",
    "R4 quedarnos con cuatro mercados", "R4. Solo US, LATAM, EU y APAC (retroalimentacion del avance 1)")

# R5: solo ventas con descuento mayor a 0
con_r5, excluidos_r5 = separar(
    con_r4, con_r4["Descuento"] > 0, "R5", "Venta sin descuento",
    "R5 quedarnos con ventas con descuento", "R5. Solo ventas con descuento mayor a 0 (indicacion de la profesora)")

print("Despues de R4:", len(con_r4), "| fuera por mercado:", len(excluidos_r4))
print(excluidos_r4["Mercado"].value_counts())
print("\nDespues de R5:", len(con_r5), "| fuera por no tener descuento:", len(excluidos_r5))

Despues de R4: 40985 | fuera por mercado: 10264
Mercado
EMEA      5298
Africa    4582
Canada     384
Name: count, dtype: int64

Despues de R5: 19141 | fuera por no tener descuento: 21844


### Paso 4. Columnas nuevas para el analisis

| Columna | ¿De donde sale? | ¿Como se calcula? | ¿Que valores puede tener? | ¿Para que la usamos? |
|---|---|---|---|---|
| Mes | Fecha Salida | El año y el mes de la venta | De 2011-01 a 2014-12 | Para ver los cambios mes a mes (subpregunta 5) |
| Tramo de descuento | Descuento ya redondeado | El rango de 10 puntos donde cae el descuento | 1 a 10 %, 11 a 20 %, 21 a 30 %, 31 a 40 %, 41 a 50 % y mas de 50 % | Para encontrar en que tramo el margen se vuelve negativo |
| Margen | Utilidad y Ventas | Utilidad ÷ Ventas de cada venta, sin restar el costo de envio (R6) | Un numero con decimales, negativo si la venta dejo perdida | Para comparar ventas de distinto tamaño |

Los tramos son los mismos del avance 1 (seccion 3.7), pero ya no esta el de 0 % porque la regla R5 solo deja ventas con descuento. Como Ventas siempre es mayor que 0, el margen nunca divide entre cero (lo revisa el paso 5).

Por la regla R6 el costo de envio lo paga el cliente, asi que no es un gasto de la tienda. Por eso la Utilidad, tal como viene en el Excel, es la utilidad neta, y el margen sale de ella sin restarle nada. El Costo Envío lo dejamos en su columna para que Logistica lo revise aparte.

In [ ]:
def derivar(tabla):
    """Crea las columnas Mes, Tramo de descuento y Margen."""
    resultado = tabla.copy()
    # Mes: año y mes, por ejemplo 2011-01
    resultado["Mes"] = resultado["Fecha Salida"].dt.to_period("M").astype("string")
    # Tramo: pd.cut pone cada descuento en su rango, por ejemplo 0.15 cae en "11 a 20 %"
    resultado["Tramo de descuento"] = pd.cut(
        resultado["Descuento"], bins=LIMITES_TRAMO, labels=NOMBRES_TRAMO).astype("string")
    # Margen de cada venta. Por la regla R6 no le restamos el Costo Envío
    resultado["Margen"] = resultado["Utilidad"] / resultado["Ventas"]

    # Revisamos que ninguna fila se quede sin tramo
    sin_tramo = int(resultado["Tramo de descuento"].isna().sum())
    anotar("derivar", len(tabla), len(resultado),
           f"Mes ({resultado['Mes'].nunique()} meses), Tramo de descuento ({resultado['Tramo de descuento'].nunique()} tramos) y Margen",
           regla="Columnas nuevas para el analisis (Nota 09, seccion 3). R6, el margen no resta el costo de envio (indicacion de la profesora)",
           resultado="Correcto" if sin_tramo == 0 else "Con observaciones",
           observaciones=f"{sin_tramo} filas sin tramo")
    return resultado


derivados = derivar(con_r5)
print("Meses:", derivados["Mes"].nunique(), "| del", derivados["Mes"].min(), "al", derivados["Mes"].max())
print(derivados["Tramo de descuento"].value_counts().reindex(NOMBRES_TRAMO))

Meses: 48 | del 2011-01 al 2014-12
Tramo de descuento
1 a 10 %       4177
11 a 20 %      6312
21 a 30 %       924
31 a 40 %      3496
41 a 50 %      2683
Mas de 50 %    1549
Name: count, dtype: Int64


#### Comprobacion de la regla R6

La profesora nos indico tomar el costo de envio como si lo pagara el cliente. Para eso necesitamos saber si la Utilidad del Excel ya trae restado el envio. Buscamos ventas del mismo producto, con la misma cantidad, el mismo descuento y el mismo monto, pero con distinto costo de envio. Si la Utilidad restara el envio, en esas ventas la Utilidad tendria que cambiar.

In [ ]:
# Columnas que deben ser iguales para decir que son la misma venta
iguales = ["ID Producto", "Cantidad", "Descuento", "Ventas"]

# True en las filas cuyo grupo de ventas iguales tiene mas de un costo de envio
envio_distinto = crudos.groupby(iguales)["Costo Envío"].transform("nunique") > 1
# Cuantas utilidades distintas hay en cada grupo
utilidades = crudos.groupby(iguales)["Utilidad"].transform("nunique")

casos = crudos[envio_distinto]
print("Ventas iguales con distinto costo de envio:", len(casos), "filas en", casos.groupby(iguales).ngroups, "grupos")
print("Filas donde la utilidad cambia dentro de su grupo:", int((utilidades[envio_distinto] > 1).sum()))

Ventas iguales con distinto costo de envio: 17786 filas en 7377 grupos
Filas donde la utilidad cambia dentro de su grupo: 0


### Paso 5. Revisar

`revisar()` le hace 14 preguntas de si o no a cada fila, como en la Nota 11 (seccion 6). Por ejemplo, la clave se repite?, el descuento esta entre 0 y 1?, el pais existe en el catalogo? Si una fila pasa todas, se queda en los datos limpios. Si falla alguna, se va a pendientes y se anota cual regla fallo.

In [ ]:
CATEGORIAS = ["Office Supplies", "Technology", "Furniture"]

def revisar(tabla, regiones, productos):
    """Regresa una tabla de True/False con cada pregunta para cada fila."""
    d = tabla["Descuento"]
    productos_catalogo = productos["ID Producto"].astype("string").str.strip().str.upper()
    controles = pd.DataFrame({
        "clave_unica": ~tabla.duplicated(subset=CLAVE, keep=False),          # pedido + producto no se repite
        "mercado_en_alcance": tabla["Mercado"].isin(MERCADOS_ALCANCE),         # es de los cuatro mercados
        "descuento_mayor_a_0_y_hasta_1": d.gt(0) & d.le(1),
        "descuento_con_dos_decimales": d.round(2).eq(d),                       # ya no hay tres decimales
        "ventas_positivas": tabla["Ventas"].gt(0),                             # asi el margen no divide entre 0
        "cantidad_entera_positiva": tabla["Cantidad"].ge(1) & tabla["Cantidad"].mod(1).eq(0),
        "fecha_en_periodo": tabla["Fecha Salida"].between("2011-01-01", "2014-12-31"),
        "entrega_no_antes_de_salida": tabla["Fecha Entrega"].ge(tabla["Fecha Salida"]),
        "categoria_conocida": tabla["Categoria"].isin(CATEGORIAS),
        "pais_en_catalogo": tabla["País"].isin(regiones["País"]),              # el pais existe en Regiones
        "producto_en_catalogo": tabla["ID Producto"].isin(productos_catalogo), # el producto existe en Productos
        "ganancia_coherente": tabla["Ganancia"].eq("Ganancia").eq(tabla["Utilidad"].ge(0)),
        "tramo_asignado": tabla["Tramo de descuento"].notna(),
        "envios_1_o_2": tabla["Número de envíos"].isin([1, 2]),
    }, index=tabla.index)
    # Si alguna respuesta quedo vacia la contamos como False, porque no podemos dar por buena una regla sin el dato
    return controles.astype("boolean").fillna(False).astype(bool)


controles = revisar(derivados, regiones, productos)
cumple = controles.all(axis=1)          # True si la fila paso las 14 preguntas
limpios = derivados[cumple].copy()
pendientes = derivados[~cumple].copy()
# A cada pendiente le anotamos que preguntas fallo
pendientes["Reglas que fallan"] = [", ".join(fila.index[~fila]) for _, fila in controles[~cumple].iterrows()]

anotar("revisar", len(derivados), len(limpios),
       f"{controles.shape[1]} preguntas por fila; {len(pendientes)} filas pendientes",
       regla="14 preguntas de si o no a cada fila (Nota 11, seccion 6)",
       resultado="Correcto" if len(pendientes) == 0 else "Con observaciones")

print("Filas que pasan cada pregunta:")
print(controles.sum().to_string())
print("\nFilas limpias:", len(limpios), "| pendientes:", len(pendientes))

Filas que pasan cada pregunta:
clave_unica                      19141
mercado_en_alcance               19141
descuento_mayor_a_0_y_hasta_1    19141
descuento_con_dos_decimales      19141
ventas_positivas                 19141
cantidad_entera_positiva         19141
fecha_en_periodo                 19141
entrega_no_antes_de_salida       19141
categoria_conocida               19141
pais_en_catalogo                 19141
producto_en_catalogo             19141
ganancia_coherente               19141
tramo_asignado                   19141
envios_1_o_2                     19141

Filas limpias: 19141 | pendientes: 0


### Paso 6. Preparar

Dejamos solo las 23 columnas que acordamos en el contrato, con el tipo correcto (por ejemplo, Cantidad como numero entero), y ordenamos por la clave.

In [ ]:
def preparar(tabla):
    """Deja la tabla con las columnas y tipos que acordamos."""
    salida = tabla[COLUMNAS_SALIDA].copy()
    salida["Cantidad"] = salida["Cantidad"].astype("Int64")                    # numero entero
    salida["Número de envíos"] = salida["Número de envíos"].astype("Int64")
    salida = salida.sort_values(CLAVE).reset_index(drop=True)                   # ordenamos por la clave
    anotar("preparar", len(tabla), len(salida),
           f"{salida.shape[1]} columnas acordadas; ordenado por {' + '.join(CLAVE)}",
           regla="Contrato de salida", resultado="Correcto")
    return salida


listos = preparar(limpios)
print(listos.shape)
display(listos.head(3))

(19141, 23)


,ID Venta,ID Producto,ID Cliente,Fecha Salida,Fecha Entrega,Mes,Mercado,País,Segmento,Categoria,Subcategoria,Nombre del producto,Método Envio,Cantidad,Ventas,Descuento,Descuento original,Tramo de descuento,Utilidad,Margen,Ganancia,Costo Envío,Número de envíos
0,AR-2011-104024,OFF-AR-10002583,CC-12145,2011-09-14,2011-09-16,2011-09,LATAM,Argentina,Consumer,Office Supplies,Art,"BIC Markers, Fluorescent",Second Class,7,77.532,0.4,0.4,31 a 40 %,-41.468,-0.534850,Perdida,6.61,1
1,AR-2011-104899,TEC-AC-10003081,KN-16705,2011-12-14,2011-12-18,2011-12,LATAM,Argentina,Home Office,Technology,Accessories,"Memorex Mouse, USB",Standard Class,4,45.408,0.4,0.4,31 a 40 %,-25.792,-0.568006,Perdida,4.27,1
2,AR-2011-105025,OFF-FA-10001237,GA-14725,2011-12-24,2011-12-31,2011-12,LATAM,Argentina,Consumer,Office Supplies,Fasteners,"Stockwell Push Pins, Bulk Pack",Standard Class,2,11.184,0.4,0.4,31 a 40 %,0.744,0.066524,Ganancia,1.35,1


### Paso 7. Guardar

Guardamos los datos limpios, los excluidos (con su regla y motivo), los pendientes y la bitacora. Cada vez que se corre el notebook estos archivos se vuelven a crear desde el Excel original.

In [ ]:
# Juntamos en una sola tabla todo lo que quedo fuera por las reglas R2, R4 y R5
excluidos = pd.concat([excluidos_r2, excluidos_r4, excluidos_r5], ignore_index=True)

def guardar(listos, excluidos, pendientes):
    """Guarda las salidas en data/processed y la bitacora en bitacora/."""
    listos.to_csv(CARPETA_SALIDAS / "ventas_preparadas.csv", index=False, date_format="%Y-%m-%d")
    excluidos.to_csv(CARPETA_SALIDAS / "registros_excluidos.csv", index=False, date_format="%Y-%m-%d")
    pendientes.to_csv(CARPETA_SALIDAS / "registros_pendientes.csv", index=False, date_format="%Y-%m-%d")
    anotar("guardar", len(listos), len(listos),
           f"ventas_preparadas.csv ({len(listos)}), registros_excluidos.csv ({len(excluidos)}), "
           f"registros_pendientes.csv ({len(pendientes)}) y bitacora.csv",
           regla="Guardar los archivos de salida", resultado="Correcto")
    pd.DataFrame(bitacora).to_csv(CARPETA_BITACORA / "bitacora.csv", index=False)


guardar(listos, excluidos, pendientes)

# Imprimimos que archivos se guardaron y cuanto pesan
for archivo in sorted(CARPETA_SALIDAS.glob("*.csv")) + [CARPETA_BITACORA / "bitacora.csv"]:
    print(archivo.relative_to(RAIZ), f"{archivo.stat().st_size / 1_000_000:.1f} MB")

data/processed/registros_excluidos.csv 9.8 MB
data/processed/registros_pendientes.csv 0.0 MB
data/processed/ventas_preparadas.csv 4.5 MB
bitacora/bitacora.csv 0.0 MB


## 3.4 Conciliacion y validacion del proceso

### Contrato de salida

Ahora revisamos la tabla limpia completa contra lo que acordamos en el contrato. Todas las respuestas deben salir `True`.

In [ ]:
contrato_salida = pd.Series({
    "Tiene las 23 columnas que acordamos, en orden?": list(listos.columns) == COLUMNAS_SALIDA,
    "Ningun pedido + producto se repite?": not listos.duplicated(subset=CLAVE).any(),
    "No hay vacios?": not listos.isna().any().any(),
    "Fecha Salida y Fecha Entrega son fechas?": all(pd.api.types.is_datetime64_any_dtype(listos[c]) for c in FECHAS),
    "Ventas, Descuento, Utilidad, Margen y Costo Envío son numeros?": all(
        pd.api.types.is_float_dtype(listos[c]) for c in ["Ventas", "Descuento", "Utilidad", "Margen", "Costo Envío"]),
    "Cantidad y Número de envíos son enteros?": all(
        pd.api.types.is_integer_dtype(listos[c]) for c in ["Cantidad", "Número de envíos"]),
    "Solo estan US, LATAM, EU y APAC?": set(listos["Mercado"]) <= set(MERCADOS_ALCANCE),
    "Todos los descuentos son mayores a 0 y hasta 1?": listos["Descuento"].between(0, 1, inclusive="right").all(),
    "Ningun descuento tiene tres decimales?": listos["Descuento"].round(2).eq(listos["Descuento"]).all(),
    "Cada fila tiene uno de los 6 tramos?": listos["Tramo de descuento"].isin(NOMBRES_TRAMO).all(),
    "Margen es igual a Utilidad / Ventas?": (listos["Margen"] - listos["Utilidad"] / listos["Ventas"]).abs().max() < 1e-12,
})
print(contrato_salida.to_string())
print("\n¿Se cumple todo el contrato?", contrato_salida.all())

Tiene las 23 columnas que acordamos, en orden?                    True
Ningun pedido + producto se repite?                               True
No hay vacios?                                                    True
Fecha Salida y Fecha Entrega son fechas?                          True
Ventas, Descuento, Utilidad, Margen y Costo Envío son numeros?    True
Cantidad y Número de envíos son enteros?                          True
Solo estan US, LATAM, EU y APAC?                                  True
Todos los descuentos son mayores a 0 y hasta 1?                   True
Ningun descuento tiene tres decimales?                            True
Cada fila tiene uno de los 6 tramos?                              True
Margen es igual a Utilidad / Ventas?                              True

¿Se cumple todo el contrato? True


### ¿A donde fue cada fila del Excel?

Cada fila limpia dice en *Número de envíos* cuantas filas del Excel representa (1, o 2 si juntamos una pareja). Por eso contamos en filas del Excel, asi cada fila se cuenta una sola vez en el lugar donde termino. La tabla `listos` sale de `limpios`, entonces no las sumamos juntas.

In [ ]:
# Todos los lugares donde puede terminar una fila
destinos = {
    "Fuera por R2 (parejas con cliente distinto)": excluidos_r2,
    "Fuera por R4 (mercado fuera del alcance)": excluidos_r4,
    "Fuera por R5 (venta sin descuento)": excluidos_r5,
    "Pendientes (no pasaron la revision)": pendientes,
    "Limpias": listos,
}
conciliacion = pd.DataFrame({
    "Filas en la tabla": {d: len(t) for d, t in destinos.items()},
    "Filas del Excel que representan": {d: int(t["Número de envíos"].sum()) for d, t in destinos.items()},
})
display(conciliacion)

# La suma de todos los destinos debe dar las filas del Excel
total_destinos = conciliacion["Filas del Excel que representan"].sum()
print(f"Filas del Excel: {len(crudos):,} | suma de los destinos: {total_destinos:,}")
print("Cuadran las filas?", total_destinos == len(crudos))

# Y los totales de dinero y cantidad tambien deben ser iguales
print("\nLos totales del Excel son iguales a la suma de los destinos?")
for col in SE_SUMAN:
    original = crudos[col].sum()
    destinos_suma = sum(t[col].sum() for t in destinos.values())
    print(f"  {col}: {original:,.2f} vs {destinos_suma:,.2f} →", round(original, 2) == round(destinos_suma, 2))

,Filas en la tabla,Filas del Excel que representan
Fuera por R2 (parejas con cliente distinto),6,6
Fuera por R4 (mercado fuera del alcance),10264,10271
Fuera por R5 (venta sin descuento),21844,21861
Pendientes (no pasaron la revision),0,0
Limpias,19141,19152


Filas del Excel: 51,290 | suma de los destinos: 51,290
Cuadran las filas? True

Los totales del Excel son iguales a la suma de los destinos?
  Cantidad: 178,312.00 vs 178,312.00 → True
  Ventas: 12,642,501.91 vs 12,642,501.91 → True
  Utilidad: 1,467,457.29 vs 1,467,457.29 → True
  Costo Envío: 1,352,820.69 vs 1,352,820.69 → True


**Resultado de la conciliacion:**
* **A donde fueron las 51,290 filas?** 6 las quitamos con la regla R2, 10,271 quedaron fuera por ser de EMEA, Africa o Canada (R4) y 21,861 por no tener descuento (R5). Las 19,152 que quedan se volvieron 19,141 filas limpias, porque 11 de las parejas que juntamos con la regla R3 estaban en este grupo.
* **Hay pendientes?** No, todas las filas pasaron las 14 preguntas.
* **Cuadra?** Si. La suma da 51,290, asi que ninguna fila se perdio ni se conto dos veces. Los totales de cantidad, ventas, utilidad y costo de envio del Excel tambien son iguales a la suma de todos los destinos.

## 3.5 Analisis estadistico descriptivo preliminar

Calculamos cinco medidas sobre los datos limpios, que son las ventas con descuento de los cuatro mercados. Las escogimos con la Nota 10, pensando en lo que necesita la pregunta, que es encontrar en que tramo de descuento la venta deja de dejar ganancia.

| # | ¿Que queremos saber? | Columnas | Medida | ¿Por que esta medida? |
|---|---|---|---|---|
| M1 | En que tramo ya son mayoria las ventas con perdida? | Tramo de descuento, Utilidad | Cuantas ventas tienen utilidad negativa y que porcentaje son | Cuenta ventas sin que pesen mas las ventas grandes |
| M2 | Cuanto margen deja cada tramo? ¿Cambia por categoria, subcategoria y mercado? | Tramo, Utilidad, Ventas, Categoria, Subcategoria, Mercado | Margen agregado, o sea la suma de la utilidad entre la suma de las ventas. Es como un promedio del margen donde las ventas grandes pesan mas | Es el indicador que planteamos en el avance 1, y asi una venta de 5 no pesa igual que una de 5,000 |
| M3 | Que margen deja una venta tipica de cada tramo y que tanto varia? | Tramo, Margen | Mediana, primer y tercer cuartil y RIC del margen de cada venta | La mediana no se mueve por los valores extremos y el RIC nos dice que tan separada esta la mitad de en medio |
| M4 | Mientras mas descuento, menos margen? | Descuento, Margen | Correlacion de Spearman, y la de Pearson para comparar | Spearman mide si una baja cuando la otra sube aunque no sea en linea recta |
| M5 | Que es lo que mas se vende en cada categoria, cuantas unidades y cuanto dinero lleva una venta tipica? | Categoria, Subcategoria, Nombre del producto, Cantidad, Ventas, Mercado, Descuento | Moda, media y mediana | La profesora nos pidio usar medidas de tendencia central. La moda es la unica que sirve para texto y la mediana no se mueve por las ventas muy grandes |

### M1. Ventas con perdida en cada tramo

In [ ]:
# Por cada tramo contamos las ventas y cuantas tienen utilidad negativa
m1 = listos.groupby("Tramo de descuento").agg(
    ventas=("ID Venta", "size"),
    con_perdida=("Utilidad", lambda u: int((u < 0).sum())),
).reindex(NOMBRES_TRAMO)
# Porcentaje de ventas con perdida y que parte del total es cada tramo
m1["% con perdida"] = (100 * m1["con_perdida"] / m1["ventas"]).round(1)
m1["% de las ventas"] = (100 * m1["ventas"] / m1["ventas"].sum()).round(1)
display(m1)

# Comprobamos que la suma de ventas sea igual al total de la tabla limpia
print("Comprobacion: las ventas suman", m1["ventas"].sum(), "y la tabla limpia tiene", len(listos))

,ventas,con_perdida,% con perdida,% de las ventas
Tramo de descuento,,,,
1 a 10 %,4177,880,21.1,21.8
11 a 20 %,6312,1487,23.6,33.0
21 a 30 %,924,575,62.2,4.8
31 a 40 %,3496,2813,80.5,18.3
41 a 50 %,2683,2587,96.4,14.0
Mas de 50 %,1549,1549,100.0,8.1


Comprobacion: las ventas suman 19141 y la tabla limpia tiene 19141


**Interpretacion M1:** Notamos que con descuentos de 1 a 10 % y de 11 a 20 %, mas o menos una de cada cinco ventas deja perdida (21.1 % y 23.6 %). Pero en el tramo de 21 a 30 % sube hasta 62.2 %, o sea que ahi ya la mayoria de las ventas pierde. De 41 a 50 % casi todas pierden, y con mas de 50 % pierden las 1,549. El tramo de 21 a 30 % tiene pocas ventas (924, el 4.8 %), por eso lo revisamos tambien con M2 y M3.

### M2. Margen agregado por tramo y por grupo

In [ ]:
def margen_agregado(tabla, grupos):
    """Suma de utilidad entre suma de ventas, en porcentaje, con cuantas ventas tiene cada grupo."""
    r = tabla.groupby(grupos).agg(ventas_n=("ID Venta", "size"),
                                  ventas=("Ventas", "sum"), utilidad=("Utilidad", "sum"))
    r["margen_%"] = (100 * r["utilidad"] / r["ventas"]).round(1)
    return r

# Margen por tramo
m2 = margen_agregado(listos, "Tramo de descuento").reindex(NOMBRES_TRAMO)
display(m2.round(2))

# Lo mismo pero separado por categoria y por mercado
print("Margen agregado (%) por categoria y tramo:")
display(margen_agregado(listos, ["Categoria", "Tramo de descuento"])["margen_%"].unstack().reindex(columns=NOMBRES_TRAMO))
print("Margen agregado (%) por mercado y tramo:")
display(margen_agregado(listos, ["Mercado", "Tramo de descuento"])["margen_%"].unstack().reindex(columns=NOMBRES_TRAMO))

,ventas_n,ventas,utilidad,margen_%
Tramo de descuento,,,,
1 a 10 %,4177,1689993.74,278599.48,16.5
11 a 20 %,6312,1773476.98,172659.57,9.7
21 a 30 %,924,366339.04,-20560.34,-5.6
31 a 40 %,3496,732524.51,-177423.77,-24.2
41 a 50 %,2683,440914.20,-203381.91,-46.1
Mas de 50 %,1549,186569.87,-201153.93,-107.8


Margen agregado (%) por categoria y tramo:


Tramo de descuento,1 a 10 %,11 a 20 %,21 a 30 %,31 a 40 %,41 a 50 %,Mas de 50 %
Categoria,,,,,,
Furniture,16.5,4.6,-6.2,-25.4,-48.2,-103.6
Office Supplies,16.0,13.6,-6.8,-24.4,-47.7,-110.1
Technology,17.1,11.7,-1.2,-22.8,-43.2,-113.9


Margen agregado (%) por mercado y tramo:


Tramo de descuento,1 a 10 %,11 a 20 %,21 a 30 %,31 a 40 %,41 a 50 %,Mas de 50 %
Mercado,,,,,,
APAC,16.3,9.8,-3.7,-22.7,-43.1,-92.3
EU,16.6,9.0,-12.4,-24.5,-52.5,-95.1
LATAM,NaN,6.3,NaN,-27.6,-65.7,-109.1
US,16.6,11.6,-10.0,-19.4,-35.7,-119.2


Para la decision queremos saber, en cada subcategoria y mercado, cual es el primer tramo donde el margen ya sale negativo. Tambien ponemos cuantas ventas hay, porque si un tramo tiene pocas ventas no nos podemos fiar tanto de el.

In [ ]:
def primer_tramo_negativo(tabla, grupo):
    """Para cada grupo, el primer tramo con margen negativo y cuantas ventas tiene."""
    # Sacamos el margen de cada grupo en cada tramo
    r = margen_agregado(tabla, [grupo, "Tramo de descuento"]).reset_index()
    # Ordenamos los tramos de menor a mayor y nos quedamos con el primero que sale negativo
    r["orden"] = r["Tramo de descuento"].map({t: i for i, t in enumerate(NOMBRES_TRAMO)})
    negativos = r[r["margen_%"] < 0].sort_values("orden").groupby(grupo).first()
    resumen = pd.DataFrame({
        "ventas del grupo": tabla.groupby(grupo).size(),
        "primer tramo con perdida": negativos["Tramo de descuento"],
        "margen % en ese tramo": negativos["margen_%"],
        "ventas en ese tramo": negativos["ventas_n"],
    })
    # Si un grupo nunca sale negativo ponemos "Ninguno"
    resumen["primer tramo con perdida"] = resumen["primer tramo con perdida"].fillna("Ninguno")
    return resumen.sort_values("ventas del grupo", ascending=False)

print("Por subcategoria:")
display(primer_tramo_negativo(listos, "Subcategoria"))
print("Por mercado:")
display(primer_tramo_negativo(listos, "Mercado"))

Por subcategoria:


,ventas del grupo,primer tramo con perdida,margen % en ese tramo,ventas en ese tramo
Subcategoria,,,,
Binders,2218,31 a 40 %,-30.0,212
Storage,2098,31 a 40 %,-27.0,380
Chairs,1968,21 a 30 %,-4.0,392
Phones,1487,31 a 40 %,-26.0,323
Furnishings,1228,31 a 40 %,-26.0,314
Bookcases,1159,21 a 30 %,-13.0,10
Paper,1156,31 a 40 %,-28.5,157
Art,1107,21 a 30 %,-6.8,98
Accessories,987,31 a 40 %,-22.7,198


Por mercado:


,ventas del grupo,primer tramo con perdida,margen % en ese tramo,ventas en ese tramo
Mercado,,,,
APAC,6307,21 a 30 %,-3.7,647
US,5193,21 a 30 %,-10.0,226
EU,3862,21 a 30 %,-12.4,51
LATAM,3779,31 a 40 %,-27.6,2080


**Interpretacion M2:**
* **Donde cambia?** El margen es positivo hasta el tramo de 11 a 20 % (9.7 %) y pasa a negativo en el de 21 a 30 % (−5.6 %). Esto pasa igual en las tres categorias y en APAC, EU y US.
* **Y en LATAM?** En LATAM no hay ventas con descuentos de 1 a 10 % ni de 21 a 30 %, entonces solo sabemos que su margen pasa de 6.3 % con 11 a 20 % a −27.6 % con 31 a 40 %.
* **Es igual en todas las subcategorias?** No. Supplies ya pierde desde 11 a 20 % (−12.2 %). Chairs, Bookcases, Art, Tables y Machines pierden desde 21 a 30 %, y Furnishings y Phones quedan casi en cero en ese tramo. Las otras 10 subcategorias no tienen ventas con descuentos de 21 a 30 %, entonces solo sabemos que su margen es positivo con 11 a 20 % y negativo con 31 a 40 %.
* **Que tanto nos podemos fiar?** Algunos grupos tienen pocas ventas, por ejemplo 10 en Bookcases y 37 en Machines en el tramo de 21 a 30 %. Esas diferencias hay que confirmarlas antes de poner un limite por subcategoria.

### M3. Margen de una venta tipica en cada tramo

In [ ]:
# describe saca el conteo y los cuartiles del margen en cada tramo
m3 = listos.groupby("Tramo de descuento")["Margen"].describe(percentiles=[0.25, 0.5, 0.75]).reindex(NOMBRES_TRAMO)
m3 = m3[["count", "25%", "50%", "75%"]].rename(columns={"count": "ventas", "25%": "Q1", "50%": "mediana", "75%": "Q3"})
m3["RIC"] = m3["Q3"] - m3["Q1"]   # RIC = Q3 − Q1

# Lo pasamos a porcentaje para leerlo mas facil
m3_porcentaje = (100 * m3[["Q1", "mediana", "Q3", "RIC"]]).round(1).add_suffix(" (%)")
m3_porcentaje.insert(0, "ventas", m3["ventas"].astype(int))
display(m3_porcentaje)

,ventas,Q1 (%),mediana (%),Q3 (%),RIC (%)
Tramo de descuento,,,,,
1 a 10 %,4177,2.2,16.6,31.1,28.9
11 a 20 %,6312,1.2,12.5,32.5,31.2
21 a 30 %,924,-19.2,-4.3,12.2,31.4
31 a 40 %,3496,-45.0,-24.6,-5.0,40.0
41 a 50 %,2683,-68.0,-45.5,-22.7,45.3
Mas de 50 %,1549,-150.0,-87.5,-70.0,80.0


**Interpretacion M3:** La venta tipica de cada tramo dice lo mismo que M2. La mediana del margen es positiva hasta el tramo de 11 a 20 % (12.5 %) y sale negativa desde el de 21 a 30 % (−4.3 %). En ese tramo el primer cuartil es −19.2 % y el tercero 12.2 %, o sea que la mitad de en medio de las ventas va desde perder hasta ganar, por eso lo vemos como un tramo de cambio. El RIC va creciendo con el descuento, de 28.9 puntos en el primer tramo a 80 en el ultimo, entonces con descuentos altos el resultado de cada venta es mucho mas variado.

### M4. Mientras hay mas descuento, hay menos margen?

In [ ]:
# Tomamos las parejas de descuento y margen de cada venta
pares = listos[["Descuento", "Margen"]].dropna()

# Spearman es la correlacion de Pearson pero calculada con la posicion (rango) de cada valor.
# La calculamos asi para no necesitar la libreria scipy.
spearman = pares["Descuento"].rank().corr(pares["Margen"].rank())
pearson = pares["Descuento"].corr(pares["Margen"], method="pearson")   # Pearson para comparar

print("Parejas usadas:", len(pares))
print("Spearman:", round(spearman, 4))
print("Pearson: ", round(pearson, 4))

Parejas usadas: 19141
Spearman: -0.7332
Pearson:  -0.8128


**Interpretacion M4:** Con las 19,141 ventas, Spearman da −0.73, que es una relacion negativa fuerte, es decir, mientras mas descuento, menos margen. Pearson da −0.81 y va en el mismo sentido. Esto solo dice que las dos columnas se mueven juntas, no prueba que el descuento cause la perdida. Tambien puede influir a que productos se les da mas descuento o costos que el archivo no trae.

### M5. Lo que mas se vende en cada categoria (media, mediana y moda)

La profesora nos pidio agregar medidas de tendencia central para responder algo concreto, aunque la guia no lo pide. Las usamos para saber que es lo que mas se vende en cada categoria y como es una venta tipica. Son los datos limpios, o sea ventas con descuento de los cuatro mercados. Cada medida responde una pregunta distinta (Nota 10, diapositivas 11 a 14).

| Medida | Que pregunta responde? | Como se obtiene? | Para que la usamos aqui? |
|---|---|---|---|
| Media | Cuanto le toca a cada venta si repartimos el total en partes iguales? | La suma entre el numero de datos, con `mean()` | Cantidad y monto promedio de una venta |
| Mediana | Que valor queda en medio? | El valor central de la lista ordenada, con `median()` | Cantidad y monto de una venta tipica, sin que la muevan las ventas muy grandes |
| Moda | Que valor se repite mas? | El valor con mas frecuencia, con `mode()`. Puede haber empates | La subcategoria y el producto que aparecen en mas ventas, y la cantidad y el descuento que mas se repiten |

La moda es la unica de las tres que sirve para texto, como el nombre de un producto. Por eso con ella sacamos lo mas vendido. Aqui "mas vendido" quiere decir que aparece en mas ventas.

In [ ]:
# La moda es el valor que mas se repite. mode() puede regresar varios si empatan,
# por eso los juntamos separados por coma
def moda(columna):
    return ", ".join(str(valor) for valor in columna.mode())

# Cuantas veces se repite la moda (lo usamos para saber cuantas ventas tiene el producto mas vendido)
def veces_de_la_moda(columna):
    return columna.value_counts().max()

# Tabla por categoria: lo mas vendido (moda) y como es una venta tipica (media, mediana y moda)
m5 = listos.groupby("Categoria").agg(
    ventas=("Nombre del producto", "size"),
    subcategoria_mas_vendida=("Subcategoria", moda),
    producto_mas_vendido=("Nombre del producto", moda),
    veces_que_aparece=("Nombre del producto", veces_de_la_moda),
    cantidad_media=("Cantidad", "mean"),
    cantidad_mediana=("Cantidad", "median"),
    cantidad_moda=("Cantidad", moda),
    venta_media=("Ventas", "mean"),
    venta_mediana=("Ventas", "median"),
)
# .T voltea la tabla para que cada categoria sea una columna y se lea mejor
print("Lo mas vendido y la venta tipica de cada categoria:")
display(m5.round(2).T)

# El descuento de cada mercado con las tres medidas, para compararlo con el limite de 20 %
descuento_mercado = listos.groupby("Mercado")["Descuento"].agg(
    media="mean", mediana="median", moda=moda).reindex(MERCADOS_ALCANCE)
print("Descuento por mercado:")
display(descuento_mercado.round(3))

Lo mas vendido y la venta tipica de cada categoria:


Categoria,Furniture,Office Supplies,Technology
ventas,4989,10301,3851
subcategoria_mas_vendida,Chairs,Binders,Phones
producto_mas_vendido,"Hon Executive Leather Armchair, Adjustable",Staples,"Hewlett Copy Machine, Color"
veces_que_aparece,34,86,26
cantidad_media,3.77,3.76,3.72
cantidad_mediana,3.0,3.0,3.0
cantidad_moda,2,2,2
venta_media,426.14,130.04,447.74
venta_mediana,241.57,42.45,252.04


Descuento por mercado:


,media,mediana,moda
Mercado,,,
US,0.300,0.20,0.2
LATAM,0.369,0.40,0.4
EU,0.267,0.15,0.1
APAC,0.259,0.17,0.1


**Interpretacion M5:**

* **Lo mas vendido.** La subcategoria que aparece en mas ventas es Chairs en Furniture, Binders en Office Supplies y Phones en Technology. Los productos mas vendidos son Hon Executive Leather Armchair, Adjustable, Staples y Hewlett Copy Machine, Color, y si contamos por unidades salen los mismos tres. Pero cada uno aparece en muy pocas ventas (34 de 4,989, 86 de 10,301 y 26 de 3,851), porque cada categoria tiene cientos de productos distintos. Ningun producto concentra las ventas, asi que tiene mas sentido poner el descuento maximo por subcategoria y no por producto.
* **Una venta tipica.** En las tres categorias la mediana de la cantidad es 3 unidades y la moda es 2. La media sale un poco mas alta (3.7) porque hay ventas de hasta 15 unidades. Con el monto la diferencia es mas grande. En Office Supplies la media es 130.04 y la mediana 42.45, o sea que unas pocas ventas grandes jalan la media hacia arriba. Por eso en M3 usamos la mediana y no la media.
* **El descuento de cada mercado.** El descuento que mas se repite es 20 % en US y 10 % en EU y APAC, que no pasan el limite de alrededor de 20 % que vimos en M2. En LATAM la moda y la mediana son 40 %, y 7 de cada 10 de sus ventas con descuento tienen 40 % o mas. Eso esta muy arriba del limite, y en M2 vimos que LATAM ya pierde con 31 a 40 % (−27.6 %). En los cuatro mercados la media es mas alta que la mediana, porque los descuentos de hasta 80 % y 85 % la jalan hacia arriba.

### Comparacion con los datos originales

La regla R1 movio de tramo algunas ventas. Para ver si eso cambia algo, sacamos el margen por tramo con los descuentos como venian (sin redondear) en los mismos cuatro mercados y lo comparamos con el de los datos limpios.

In [ ]:
# Tomamos los datos como venian, de los cuatro mercados y con descuento
originales = crudos[crudos["Mercado"].isin(MERCADOS_ALCANCE) & (crudos["Descuento"] > 0)].copy()
# Les ponemos tramo con el descuento original
originales["Tramo de descuento"] = pd.cut(originales["Descuento"], bins=LIMITES_TRAMO, labels=NOMBRES_TRAMO).astype("string")

# Comparamos ventas y margen por tramo, antes y despues
comparacion = pd.DataFrame({
    "ventas antes": originales.groupby("Tramo de descuento").size(),
    "margen % antes": margen_agregado(originales, "Tramo de descuento")["margen_%"],
    "ventas despues": listos.groupby("Tramo de descuento").size(),
    "margen % despues": m2["margen_%"],
}).reindex(NOMBRES_TRAMO)
display(comparacion)

,ventas antes,margen % antes,ventas despues,margen % despues
Tramo de descuento,,,,
1 a 10 %,4645,17.2,4177,16.5
11 a 20 %,6274,9.9,6312,9.7
21 a 30 %,967,-5.5,924,-5.6
31 a 40 %,3392,-23.8,3496,-24.2
41 a 50 %,2788,-45.3,2683,-46.1
Mas de 50 %,1549,-107.8,1549,-107.8


**Interpretacion de la comparacion:** Con los descuentos como venian, las 461 ventas de copiadoras con 0.002 contaban como descuento de 1 a 10 %, y las de 0.202 y 0.402 caian un tramo mas arriba. Despues de redondear, el margen de cada tramo cambia menos de un punto (por ejemplo, de 17.2 % a 16.5 % en el primer tramo y de −5.5 % a −5.6 % en el de 21 a 30 %). El tramo donde el margen sale negativo sigue siendo el de 21 a 30 %, asi que la regla R1 no cambia la conclusion.

## 3.6 Bitacora automatica y trazabilidad

Esta es la bitacora que fue llenando la funcion `anotar()` en esta corrida. Se guarda en `bitacora/bitacora.csv` y se vuelve a escribir cada vez que se corre el notebook, por eso no se edita a mano. La bitacora del equipo, con las tareas, quien las hizo y los cambios por retroalimentacion, esta aparte en `bitacora/bitacora_equipo.md`.

In [ ]:
# Leemos la bitacora que se guardo y mostramos las columnas mas importantes
bitacora_csv = pd.read_csv(CARPETA_BITACORA / "bitacora.csv", keep_default_na=False)
display(bitacora_csv[["paso", "filas_antes", "filas_despues", "regla", "resultado", "detalle", "observaciones"]])
print("Campos de la bitacora:", list(bitacora_csv.columns))

,paso,filas_antes,filas_despues,regla,resultado,detalle,observaciones
0,extraer,51290,51290,Revisar la huella y que esten las 35 columnas,Correcto,Hoja Ventas Supermercado con 35 columnas; vent...,Es la misma version del avance 2
1,normalizar,51290,51290,"Quitar espacios, pasar codigos a mayusculas y ...",Correcto,139 codigos y 16 nombres de producto corregido...,
2,R1 redondear descuentos,51290,51290,R1. Los descuentos con tres decimales se toman...,Correcto,629 descuentos redondeados a dos decimales,Todos son Copiers de LATAM
3,R2 quitar parejas con cliente distinto,51290,51284,R2. Las parejas con cliente distinto no son el...,Correcto,6 filas de 3 parejas pasan a excluidos,
4,R3 juntar envios,51284,51249,R3. Las parejas se toman como un pedido mandad...,Correcto,"35 parejas juntadas; se sumaron Cantidad, Vent...",En las parejas solo cambian las cuatro columna...
5,R4 quedarnos con cuatro mercados,51249,40985,"R4. Solo US, LATAM, EU y APAC (retroalimentaci...",Correcto,10264 filas pasan a excluidos (10271 filas del...,
6,R5 quedarnos con ventas con descuento,40985,19141,R5. Solo ventas con descuento mayor a 0 (indic...,Correcto,21844 filas pasan a excluidos (21861 filas del...,
7,derivar,19141,19141,"Columnas nuevas para el analisis (Nota 09, sec...",Correcto,"Mes (48 meses), Tramo de descuento (6 tramos) ...",0 filas sin tramo
8,revisar,19141,19141,"14 preguntas de si o no a cada fila (Nota 11, ...",Correcto,14 preguntas por fila; 0 filas pendientes,
9,preparar,19141,19141,Contrato de salida,Correcto,23 columnas acordadas; ordenado por ID Venta +...,


Campos de la bitacora: ['fecha', 'archivo', 'paso', 'filas_antes', 'filas_despues', 'detalle', 'responsable', 'regla', 'resultado', 'observaciones']


## 3.7 Reproducibilidad, limitaciones y cierre

### Reproducibilidad

Volvemos a correr todo el proceso desde el Excel con una sola funcion, sin guardar nada, y lo comparamos con la tabla limpia que ya guardamos. Si salen identicas quiere decir que con el mismo archivo y las mismas reglas siempre sale lo mismo, aunque cambie la fecha de la bitacora.

In [ ]:
def ejecutar_etl(ruta):
    """Corre todo el proceso desde el Excel y regresa la tabla limpia."""
    bitacora.clear()   # empezamos la bitacora desde cero
    # Mismos pasos y en el mismo orden que arriba
    crudos_, regiones_, productos_, _ = extraer(ruta)
    tabla = r1_redondear_descuentos(normalizar(crudos_))
    tabla, _ = r2_eliminar_parejas_inconsistentes(tabla)
    tabla = r3_unir_envios(tabla)
    tabla, _ = separar(tabla, tabla["Mercado"].isin(MERCADOS_ALCANCE), "R4", "Mercado fuera del alcance",
                       "R4 quedarnos con cuatro mercados", "R4. Solo US, LATAM, EU y APAC")
    tabla, _ = separar(tabla, tabla["Descuento"] > 0, "R5", "Venta sin descuento",
                       "R5 quedarnos con ventas con descuento", "R5. Solo ventas con descuento mayor a 0")
    tabla = derivar(tabla)
    controles_ = revisar(tabla, regiones_, productos_)
    return preparar(tabla[controles_.all(axis=1)])


repetida = ejecutar_etl(ENTRADA)

# Comparamos la tabla de esta corrida con la que ya guardamos
huella_guardada = hashlib.sha256((CARPETA_SALIDAS / "ventas_preparadas.csv").read_bytes()).hexdigest()
huella_repetida = hashlib.sha256(repetida.to_csv(index=False, date_format="%Y-%m-%d").encode("utf-8")).hexdigest()
print("La segunda corrida da la misma tabla que la primera?", repetida.equals(listos))
print("Es igual al CSV guardado, byte por byte?", huella_guardada == huella_repetida)

# Revisamos tambien que el Excel no haya cambiado
print("El Excel original sigue igual?", hashlib.sha256(ENTRADA.read_bytes()).hexdigest() == huella_entrada)

La segunda corrida da la misma tabla que la primera? True
Es igual al CSV guardado, byte por byte? True
El Excel original sigue igual? True


### Limitaciones

1. **Solo una parte de los datos.** Los datos limpios representan 19,152 de las 51,290 filas del Excel (37 %). Lo que concluimos vale para ventas con descuento de US, LATAM, EU y APAC.
2. **No comparamos contra ventas sin descuento.** Por la regla R5 no entran las ventas sin descuento, pero no se perdieron, estan en `registros_excluidos.csv` por si se necesitan.
3. **Cosas que el archivo no dice.** No sabemos en que moneda estan los montos (avance 1). Lo del envio si lo revisamos, y la Utilidad no lo trae restado (seccion 3.3, paso 4). Por la regla R6 tomamos el envio como un pago del cliente. Si lo pagara la tienda, el margen bajaria casi 11 puntos y la perdida empezaria desde el tramo de 11 a 20 % (−1.1 %).
4. **Reglas que suponen algo.** R1 toma los 629 descuentos raros como error de captura y R3 toma las parejas como envios en dos partes, porque asi nos lo indico la profesora. En 9 de las 35 parejas las dos filas tenian la misma cantidad. Si en realidad fueran capturas dobles, las ventas saldrian 1,477.57 mas altas (0.01 %).
5. **Grupos con pocas ventas.** Algunas combinaciones de subcategoria o mercado con tramo tienen pocas ventas o ninguna. Por ejemplo, LATAM no tiene ventas de 1 a 10 % ni de 21 a 30 %.
6. **Que se muevan juntas no quiere decir que una cause la otra.** La correlacion de M4 no prueba que el descuento provoque la perdida.

### Siguientes pasos

1. Cargar `ventas_preparadas.csv` en PostgreSQL y diseñar el modelo (unidad 3).
2. Sacar el limite de descuento por subcategoria y mercado, juntando los grupos que tienen pocas ventas.
3. Revisar como cambia todo mes a mes (subpregunta 5) con la columna Mes.